# Famous CNN Architectures — Build, Train & Compare

**Course:** ML, Deep Learning & Computer Vision — Phase 4  
**Dataset:** CIFAR-10 (auto-downloads via `torchvision`)  
**Estimated time:** 2–3 hours  

---

In this notebook you'll **implement from scratch** and **train** the architectures that changed deep learning:

| # | Architecture | Year | What you'll learn |
|---|-------------|------|-------------------|
| 1 | LeNet-5 | 1998 | The original CNN pipeline |
| 2 | AlexNet (mini) | 2012 | ReLU + Dropout + deeper |
| 3 | VGG-11 (mini) | 2014 | Stacking 3×3 convs |
| 4 | ResNet-18 (from scratch) | 2015 | Skip connections |
| 5 | ResNet-18 (pretrained) | 2015 | Transfer learning |
| 6 | EfficientNet-B0 (pretrained) | 2019 | Modern efficient design |

All models are adapted for CIFAR-10 (32×32 images, 10 classes).  
No downloads needed — everything comes from `torchvision`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import time
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

# Warn if no GPU
if device.type == 'cpu':
    print("⚠️  No GPU detected. Training will be slow.")
    print("   Consider using Google Colab (free GPU) or Kaggle Notebooks.")

---
## Data: CIFAR-10

10 classes, 50K training + 10K test images, 32×32 RGB.  
Downloads automatically on first run (~170 MB).

In [ ]:
# Data augmentation for training, plain normalisation for test
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_set = datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
test_set = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

CLASSES = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Train: {len(train_set):,} | Test: {len(test_set):,} | Classes: {len(CLASSES)}")

# Show samples
def unnorm(img):
    m = torch.tensor([0.4914,0.4822,0.4465]).view(3,1,1)
    s = torch.tensor([0.2470,0.2435,0.2616]).view(3,1,1)
    return (img*s+m).clamp(0,1)

fig, axes = plt.subplots(1, 10, figsize=(15, 1.8))
for i in range(10):
    img, label = test_set[i*1000]
    axes[i].imshow(unnorm(img).permute(1,2,0).numpy())
    axes[i].set_title(CLASSES[label], fontsize=8); axes[i].axis('off')
plt.suptitle('CIFAR-10 samples', fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()

---
## Training engine

One function to train and evaluate any model — we'll reuse it for every architecture.

In [ ]:
def train_and_evaluate(model, name, n_epochs=15, lr=1e-3):
    """
    Train a model on CIFAR-10 and return the history.
    Uses Adam + cosine annealing + weight decay.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"  Params: {total_params:,} (trainable: {trainable:,})")
    print(f"  Training for {n_epochs} epochs with Adam (lr={lr})")
    print(f"{'='*60}")
    
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_acc = 0
    t0 = time.time()
    
    for epoch in range(1, n_epochs+1):
        # Train
        model.train()
        running_loss, correct, total = 0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()*len(y)
            correct += (out.argmax(1)==y).sum().item()
            total += len(y)
        train_loss = running_loss/total
        train_acc = correct/total
        
        # Evaluate
        model.eval()
        running_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                loss = criterion(out, y)
                running_loss += loss.item()*len(y)
                correct += (out.argmax(1)==y).sum().item()
                total += len(y)
        val_loss = running_loss/total
        val_acc = correct/total
        
        scheduler.step()
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        if val_acc > best_acc: best_acc = val_acc
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:2d}/{n_epochs} | Train: {train_acc:.1%} | Val: {val_acc:.1%} | Loss: {val_loss:.4f}")
    
    elapsed = time.time() - t0
    print(f"  ✓ Done in {elapsed:.0f}s | Best val: {best_acc:.2%}")
    
    return {'name': name, 'history': history, 'best_acc': best_acc,
            'params': total_params, 'time': elapsed}

# Store all results
all_results = {}
print("Training engine ready.")

---
# 1. LeNet-5 (1998) — the original

The first successful CNN, designed by Yann LeCun for handwritten digit recognition.  
We adapt it for CIFAR-10 (colour images, 10 classes).

**Architecture:** Conv5×5(6) → Pool → Conv5×5(16) → Pool → FC(120) → FC(84) → FC(10)  
**Original:** used Sigmoid/Tanh and average pooling. We use ReLU and max pooling (modern version).

In [ ]:
class LeNet5(nn.Module):
    """LeNet-5 adapted for CIFAR-10 (32×32 RGB)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),       # (6, 28, 28)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                   # (6, 14, 14)
            nn.Conv2d(6, 16, kernel_size=5),       # (16, 10, 10)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                   # (16, 5, 5)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                          # 16*5*5 = 400
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

result = train_and_evaluate(LeNet5(), 'LeNet-5 (1998)', n_epochs=15, lr=1e-3)
all_results['LeNet-5'] = result

---
# 2. AlexNet-mini (2012) — the revolution

AlexNet was designed for 227×227 ImageNet images. We build a scaled-down version for 32×32 CIFAR-10 that preserves the key innovations: **ReLU**, **Dropout**, and deeper architecture.

Original used 11×11 and 5×5 convs — we use smaller kernels since our images are tiny.

In [ ]:
class AlexNetMini(nn.Module):
    """AlexNet-style architecture scaled for CIFAR-10 (32×32)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Original used 11×11, we use 3×3 (image is only 32×32)
            nn.Conv2d(3, 64, 3, padding=1),       # (64, 32, 32)
            nn.ReLU(inplace=True),                 # ← AlexNet's big innovation
            nn.MaxPool2d(2, 2),                    # (64, 16, 16)
            
            nn.Conv2d(64, 192, 3, padding=1),      # (192, 16, 16)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                    # (192, 8, 8)
            
            nn.Conv2d(192, 384, 3, padding=1),     # (384, 8, 8)
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, 3, padding=1),     # (256, 8, 8)
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),     # (256, 8, 8)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                    # (256, 4, 4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                           # 256*4*4 = 4096
            nn.Dropout(0.5),                        # ← AlexNet's other innovation
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

result = train_and_evaluate(AlexNetMini(), 'AlexNet-mini (2012)', n_epochs=15, lr=1e-3)
all_results['AlexNet-mini'] = result

---
# 3. VGG-11 mini (2014) — depth with only 3×3

VGG's insight: use **only 3×3 convolutions** and go deeper. Two 3×3 = one 5×5 receptive field, but fewer params and more ReLUs.

We implement VGG-11 (the smallest VGG variant) with reduced channel counts for CIFAR-10.

In [ ]:
def make_vgg_block(in_ch, out_ch, n_convs):
    """VGG block: stack n 3×3 convs + BN + ReLU, then MaxPool."""
    layers = []
    for i in range(n_convs):
        layers += [
            nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),  # VGG didn't use BN, but we add it (modern practice)
            nn.ReLU(inplace=True),
        ]
    layers.append(nn.MaxPool2d(2, 2))
    return nn.Sequential(*layers)

class VGG11Mini(nn.Module):
    """VGG-11 with reduced channels for CIFAR-10."""
    def __init__(self, num_classes=10):
        super().__init__()
        # VGG-11 config: [1, 1, 2, 2, 2] conv blocks
        # Original channels: 64, 128, 256, 512, 512
        # We halve them for CIFAR-10
        self.features = nn.Sequential(
            make_vgg_block(3,   32, 1),    # (32, 16, 16)
            make_vgg_block(32,  64, 1),    # (64, 8, 8)
            make_vgg_block(64,  128, 2),   # (128, 4, 4)
            make_vgg_block(128, 256, 2),   # (256, 2, 2)
            make_vgg_block(256, 256, 2),   # (256, 1, 1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

result = train_and_evaluate(VGG11Mini(), 'VGG-11 mini (2014)', n_epochs=15, lr=1e-3)
all_results['VGG-11 mini'] = result

---
# 4. ResNet-18 from scratch (2015) — skip connections

The architecture that made 100+ layer networks possible. We build it from scratch so you see every skip connection.

**Key:** `output = F(x) + x` — the identity shortcut.

In [ ]:
class ResidualBlock(nn.Module):
    """Basic residual block: two 3×3 convs + skip connection."""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        # Main path: two 3×3 convolutions
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection: identity if shapes match, 1×1 conv if they don't
        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            # Need to match dimensions with 1×1 conv
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
    
    def forward(self, x):
        identity = self.skip(x)       # skip connection (maybe with 1×1 conv)
        
        out = F.relu(self.bn1(self.conv1(x)))  # conv → BN → ReLU
        out = self.bn2(self.conv2(out))         # conv → BN (no ReLU yet!)
        
        out = out + identity                     # ← THE skip connection
        out = F.relu(out)                        # ReLU AFTER addition
        return out


class ResNet18(nn.Module):
    """ResNet-18 for CIFAR-10 (adapted: uses 3×3 first conv instead of 7×7)."""
    
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Initial convolution (CIFAR: 3×3 instead of ImageNet's 7×7)
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        # No MaxPool (image is already 32×32, too small to pool early)
        
        # 4 residual stages (ResNet-18: [2, 2, 2, 2] blocks)
        self.layer1 = self._make_layer(64,  64,  n_blocks=2, stride=1)  # 32×32
        self.layer2 = self._make_layer(64,  128, n_blocks=2, stride=2)  # 16×16
        self.layer3 = self._make_layer(128, 256, n_blocks=2, stride=2)  # 8×8
        self.layer4 = self._make_layer(256, 512, n_blocks=2, stride=2)  # 4×4
        
        # Global average pooling + FC
        self.avgpool = nn.AdaptiveAvgPool2d(1)  # 4×4 → 1×1
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        layers = [ResidualBlock(in_ch, out_ch, stride)]  # first block may downsample
        for _ in range(1, n_blocks):
            layers.append(ResidualBlock(out_ch, out_ch, stride=1))  # rest keep same size
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)    # (64, 32, 32)
        x = self.layer1(x)   # (64, 32, 32)
        x = self.layer2(x)   # (128, 16, 16)
        x = self.layer3(x)   # (256, 8, 8)
        x = self.layer4(x)   # (512, 4, 4)
        x = self.avgpool(x)  # (512, 1, 1)
        x = x.flatten(1)     # (512,)
        x = self.fc(x)       # (10,)
        return x

# Verify shapes
model_test = ResNet18()
with torch.no_grad():
    test_out = model_test(torch.randn(2, 3, 32, 32))
print(f"ResNet-18 forward: (2, 3, 32, 32) → {tuple(test_out.shape)}")
print(f"Parameters: {sum(p.numel() for p in model_test.parameters()):,}")

result = train_and_evaluate(ResNet18(), 'ResNet-18 from scratch (2015)', n_epochs=15, lr=1e-3)
all_results['ResNet-18 (scratch)'] = result

---
# 5. ResNet-18 pretrained — transfer learning

Load a ResNet-18 pretrained on ImageNet (1.2M images), replace the last layer, and fine-tune.  
This is what you'd do in a real project.

In [ ]:
# Load pretrained weights (downloads ~44MB on first run)
resnet_pretrained = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace the final FC layer for 10 classes
resnet_pretrained.fc = nn.Linear(resnet_pretrained.fc.in_features, 10)

# The first conv is 7×7 stride 2 (designed for 224×224)
# For CIFAR-10 (32×32), we replace it with 3×3 stride 1 to avoid losing too much spatial info
resnet_pretrained.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
resnet_pretrained.maxpool = nn.Identity()  # skip the maxpool (too aggressive for 32×32)

result = train_and_evaluate(resnet_pretrained, 'ResNet-18 pretrained (fine-tuned)',
                            n_epochs=15, lr=5e-4)
all_results['ResNet-18 (pretrained)'] = result

---
# 6. EfficientNet-B0 pretrained (2019) — modern & efficient

EfficientNet uses **compound scaling** (depth × width × resolution) and depthwise separable convolutions.  
B0 is the smallest variant — only 5.3M params but very strong accuracy.

In [ ]:
# Load pretrained EfficientNet-B0 (downloads ~20MB on first run)
effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Replace classifier head
effnet.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(effnet.classifier[1].in_features, 10),
)

result = train_and_evaluate(effnet, 'EfficientNet-B0 pretrained (2019)',
                            n_epochs=15, lr=3e-4)
all_results['EfficientNet-B0'] = result

---
# Head-to-head comparison

In [ ]:
# === COMPARISON PLOT ===
colors = {
    'LeNet-5': '#7F77DD',
    'AlexNet-mini': '#D85A30',
    'VGG-11 mini': '#378ADD',
    'ResNet-18 (scratch)': '#D4537E',
    'ResNet-18 (pretrained)': '#1D9E75',
    'EfficientNet-B0': '#EF9F27',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Validation accuracy curves
for name, res in all_results.items():
    c = colors.get(name, '#888')
    h = res['history']
    epochs = range(1, len(h['val_acc'])+1)
    axes[0].plot(epochs, h['val_acc'], 'o-', color=c, linewidth=2, markersize=3,
                label=f"{name} ({res['best_acc']:.1%})")

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Validation accuracy', fontsize=12)
axes[0].set_title('Convergence comparison', fontweight='bold', fontsize=13)
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(True, alpha=0.2)

# Bar chart: best accuracy vs params
names = list(all_results.keys())
accs = [all_results[n]['best_acc'] for n in names]
params = [all_results[n]['params'] for n in names]
bar_colors = [colors.get(n, '#888') for n in names]

axes[1].barh(range(len(names)), accs, color=bar_colors, height=0.6)
axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels([f"{n}\n({all_results[n]['params']/1e6:.1f}M params)" for n in names], fontsize=8)
axes[1].set_xlabel('Best validation accuracy', fontsize=12)
axes[1].set_title('Best accuracy (with param count)', fontweight='bold', fontsize=13)
for i, (a, n) in enumerate(zip(accs, names)):
    axes[1].text(a + 0.003, i, f'{a:.1%}', va='center', fontsize=9, fontweight='bold')
axes[1].set_xlim(0.5, 1.0)
axes[1].grid(True, alpha=0.2, axis='x')

plt.suptitle('CNN Architecture Comparison on CIFAR-10', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === SUMMARY TABLE ===
print(f"\n{'Architecture':30s} {'Params':>10s} {'Best Acc':>10s} {'Time':>8s} {'Acc/param':>12s}")
print("─" * 75)

for name in all_results:
    r = all_results[name]
    efficiency = r['best_acc'] / (r['params'] / 1e6)  # accuracy per million params
    print(f"{name:30s} {r['params']/1e6:>8.1f}M {r['best_acc']:>10.2%} {r['time']:>7.0f}s {efficiency:>10.2f}%/M")

# Find winner in each category
best_acc_name = max(all_results, key=lambda k: all_results[k]['best_acc'])
best_eff_name = max(all_results, key=lambda k: all_results[k]['best_acc']/(all_results[k]['params']/1e6))
fastest_name = min(all_results, key=lambda k: all_results[k]['time'])

print(f"\n🏆 Highest accuracy:    {best_acc_name} ({all_results[best_acc_name]['best_acc']:.2%})")
print(f"⚡ Most efficient:      {best_eff_name} (best acc per million params)")
print(f"🚀 Fastest training:    {fastest_name} ({all_results[fastest_name]['time']:.0f}s)")

---
## What you should see

| Architecture | Expected accuracy | Why |
|-------------|------------------|-----|
| **LeNet-5** | ~60–65% | Too small, only 60K params, can't capture CIFAR complexity |
| **AlexNet-mini** | ~78–83% | Deeper + dropout helps, but FC layers waste params |
| **VGG-11 mini** | ~85–88% | Depth + BatchNorm works well |
| **ResNet-18 scratch** | ~88–92% | Skip connections enable effective training |
| **ResNet-18 pretrained** | ~90–93% | ImageNet features transfer beautifully |
| **EfficientNet-B0** | ~91–95% | Modern architecture + pretrained = best |

## Key lessons

1. **Depth matters** — LeNet (5 layers) → VGG (16 layers) → ResNet (18+ layers): each step improved accuracy
2. **Skip connections are essential** — ResNet from scratch beats VGG despite similar depth, because gradients flow better
3. **Transfer learning is almost always the right choice** — pretrained models beat from-scratch even with 15 epochs
4. **Efficiency matters** — EfficientNet gets great accuracy with far fewer params than VGG
5. **Architecture + training tricks compound** — BatchNorm, data augmentation, cosine LR all help every architecture

---
## Experiments to try

1. **Train longer:** Run each model for 50–100 epochs. The ranking may shift.
2. **Remove data augmentation:** Rerun without `RandomHorizontalFlip` and `RandomCrop`. Which models overfit most?
3. **Freeze pretrained features:** For ResNet/EfficientNet, set `requires_grad=False` for everything except the last layer. How does accuracy change?
4. **Try more architectures:** `models.resnet50()`, `models.mobilenet_v3_small()`, `models.vit_b_16()` — all one-liners in torchvision.
5. **Different dataset:** Replace CIFAR-10 with `datasets.FashionMNIST` or `datasets.SVHN` — do the rankings change?